# Two sequential optimisers over one graph

PROCESS poses one optimisation problem over all of a run's iteration variables and
solves it with VMCON. This case splits it into **two problems solved one after the
other** on the same models. A *magnet* stage sizes the coil: its own objective, a subset
of the iteration variables, and the constraints that depend only on those. A *plasma*
stage then finds the operating point: the file's own objective, the remaining variables
and constraints. No iteration variable belongs to both.

The graph decides whether such a split is legal. Both problems are inserted and the run
order is derived. The stages are sequential only if no constraint of the first depends
on a variable of the second. Otherwise the two problems end up in one loop, and the
blocking refuses it; one problem over both would be SAND. Inside each stage, the
feedback loops that fall in that stage are folded into its optimiser, as SAND does for
the whole machine.

The full experiment, with results and discussion, is in
[`two_driver_report.md`](two_driver_report.md). The scripts behind it are in
[`scripts/`](scripts/README.md). This notebook shows the configuration, the run order it
produces, and one run. The first solve includes compilation.

In [1]:
import os, sys, time
from pathlib import Path

HERE = Path.cwd()                                   # the notebook's own folder
REPO = next(p for p in (HERE, *HERE.parents) if (p / "functional_process").is_dir())
os.chdir(REPO)                                      # input files are named relative to PROCESS/
# The editable cottax checkout beside this repo, when the layout is the documented one
# (`two_opt_driver/scripts/README.md`), ahead of any installed copy.
_jaxgraph = REPO.parent.parent / "jaxgraph" / "src"
for entry in (str(REPO), str(_jaxgraph)):
    if Path(entry).is_dir() and entry not in sys.path:
        sys.path.insert(0, entry)

import jax
jax.config.update("jax_enable_x64", True)          # PROCESS is float64 throughout
import cottax
import numpy as np


print("repo  :", REPO)
print("cottax:", Path(cottax.__file__).parent)

repo  : /home/wrutten/projects/functional_PROCESS/PROCESS
cottax: /home/wrutten/projects/jaxgraph/src/cottax


## 1. The models, and the problem to split

`mda_env` builds the graph of the Helias stellarator's models with the feedback loops
cut, as in the SAND notebook's step 1. It also converges the graph once from the input
file's values, so the stages can start from a consistent machine. Then the file's
constraints and objective (COE, figure of merit 6) are built as nodes, plus a second
objective for the magnet stage: figure of merit 1, the major radius.

In [2]:
INPUT = "tests/regression/input_files/stellarator_helias.IN.DAT"

from cottax.blocking import Blocking
from cottax.evaluation.schedule import Call, Drive, Schedule
from cottax.names import PathMap, prefix_path
from cottax.nodes import ImplementedFunction
from cottax.plan import Insert, Plan
from cottax.problem import Optimise, is_fixed_point, is_optimise
from cottax.rewrites import Combine, Residualise
from cottax.spec import NodePath, VarPath
from jax.tree_util import GetAttrKey

from functional_process.architecture_examples.notebook_tools import print_recipe
from functional_process.cottax import native
from functional_process.cottax.indat import graph_for, machine_from_indat, objective_selection, switch_values_from_indat
from functional_process.cottax.mda import assign_drivers, default_drivers
from functional_process.cottax.queries import declared
from functional_process.cottax.sand import COND, constraint_nodes, design_bounds, iteration_variable_path, objective_nodes
from functional_process.cottax.sand_harness import mda_env

ref = native.native_reference(INPUT)          # ixc, icc, bounds, cold values -- PROCESS-free
machine_graph = graph_for(machine_from_indat(INPUT))
sw = switch_values_from_indat(INPUT)

g, env = mda_env(ref, graph=machine_graph)    # the cut graph, and one converged MDA on it

cnodes, equalities, inequalities, _ = constraint_nodes(g, ref.icc, ref.n_equality, sw)
obj_nodes, coe = objective_nodes(g, objective_selection(ref.i_figure_merit), sw)                  # COE, FoM 6
rmajor_nodes, rmajor_obj = objective_nodes(g, objective_selection(1), sw, label="_rmajor")        # rmajor, FoM 1
obj_nodes.update(rmajor_nodes)

cond = lambda cid: prefix_path(VarPath((GetAttrKey("constraints"), GetAttrKey(f"c{cid}"))), COND)
EQ = set(ref.icc[:ref.n_equality])
print(f"ixc = {ref.ixc}")
print(f"icc = {ref.icc}  ({ref.n_equality} equalities), objective: figure of merit {ref.i_figure_merit}")
print(f"{len(g.nodes)} nodes in the MDA graph; {len(cnodes)} constraint nodes and {len(obj_nodes)} objective nodes to insert")

ixc = [2, 3, 4, 6, 10, 56, 59, 109]
icc = [2, 16, 24, 8, 17, 18, 67, 82, 83, 62, 32, 34, 35, 65]  (2 equalities), objective: figure of merit 6
154 nodes in the MDA graph; 14 constraint nodes and 2 objective nodes to insert


Which iteration variables can a split separate? For each constraint and objective,
follow the models it depends on back to the iteration variables they read
(`scripts/coupling_probe.py`). A constraint that depends on a variable must be in the
same stage as that variable or in a later one.

In [3]:
g2 = (Plan(g) + Insert(PathMap({**cnodes, **obj_nodes}.items()))).graph
design = {iteration_variable_path(i): i for i in ref.ixc}

def reaches(node):
    cone = set(g2.ancestors([node])) | {node}
    return {design[r] for n in cone for r in g2[n].reads if r in design}

rows = [(n.leaf.name, reaches(n)) for n in g2.nodes if n.leaf.name.startswith(("Constraint", "Objective"))]
rows.sort(key=lambda r: (not r[0].startswith("Objective"), int(r[0][10:]) if r[0][10:].isdigit() else 0))
width = max(len(r[0]) for r in rows)
print(f"{'':{width}} " + "".join(f"{i:>5}" for i in ref.ixc))
for leaf, hit in rows:
    kind = "eq" if leaf.startswith("Constraint") and int(leaf[10:]) in EQ else ("obj" if leaf.startswith("Objective") else "")
    print(f"{leaf:{width}} " + "".join("  X  " if i in hit else "  .  " for i in ref.ixc), kind)

                     2    3    4    6   10   56   59  109
Objective          X    X    X    X    X    X    X    X   obj
Objective_rmajor   .    X    .    .    .    .    .    .   obj
Constraint2        X    X    X    X    X    .    .    X   eq
Constraint8        X    X    X    X    .    .    .    X   
Constraint16       X    X    X    X    .    X    X    X   eq
Constraint17       X    X    X    X    .    .    .    X   
Constraint18       X    X    X    X    .    .    X    X   
Constraint24       X    X    X    X    .    .    .    X   
Constraint32       X    X    .    .    .    .    X    .   
Constraint34       X    X    .    .    .    X    X    .   
Constraint35       X    X    .    .    .    X    X    .   
Constraint62       X    X    X    X    X    .    .    X   
Constraint65       X    X    .    .    .    X    X    .   
Constraint67       X    X    X    X    .    .    .    X   
Constraint82       X    X    .    .    .    .    X    .   
Constraint83       X    X    .    .    .    .  

Every constraint depends on `b_t` (2) and `rmajor` (3). The TF-conductor constraints
32/34/35/65/82/83 depend only on `{2, 3, 56, 59}`. COE and the net-electric equality 16
depend on all eight. So the magnet stage can own `{2, 3, 56, 59}` and answer the
conductor constraints, and the plasma stage owns the rest.

## 2. The recipe

One optimisation problem per stage: the iteration variables it owns, the objective it
minimises, the constraint ids it answers. The magnet stage also gets `b_t >= 5.0 T` as an
extra constraint. Without it, the magnet stage drives `b_t` to its lower bound and leaves
the plasma stage no feasible point; that is the report's first unexpected result.

Then, on the cut graph:

1. insert the constraint and objective nodes and the two problems;
2. `Residualise` every fixed-point requirement (`x = g(x)` becomes `g(x) - x = 0`);
3. in each loop that holds an optimiser and other problems, `Combine` those problems
   into that optimiser (SAND per stage rather than once).

Nothing here says which stage runs first: the run order is derived.

In [4]:
import equinox as eqx

class BtMin(eqx.Module):
    # `b_t >= bt_min` in cottax's convention (g <= 0 satisfied): violated when b_t < bt_min
    bt_min: float = eqx.field(static=True)
    def __call__(self, bt):
        return (self.bt_min - bt) / self.bt_min

BT_MIN = 5.0
cnodes[NodePath((GetAttrKey("ConstraintBtMin"),))] = ImplementedFunction(
    reads=(iteration_variable_path(2),), owns=(cond("bt_min"),), fn=BtMin(BT_MIN))

# stage -> (iteration variables it owns, objective, constraints it answers)
SPLIT = {
    "OptMagnet": ([2, 3, 56, 59], rmajor_obj, [32, 34, 35, 65, 82, 83, "bt_min"]),
    "OptPlasma": ([4, 6, 10, 109], coe,       [2, 16, 8, 17, 18, 24, 62, 67]),
}

def build(split, fold=True):
    # the recipe for `split`, as a `Plan` on the cut graph
    nodes = dict(cnodes); nodes.update(obj_nodes)
    for name, (ixc, objective, cids) in split.items():
        nodes[NodePath((GetAttrKey(name),))] = Optimise(
            objective=objective,
            unknowns=tuple(iteration_variable_path(i) for i in ixc),
            equalities=tuple(cond(c) for c in cids if c in EQ),
            inequalities=tuple(cond(c) for c in cids if c not in EQ),
        )
    plan = Plan(g) + Insert(PathMap(nodes.items()))                        # 1
    for p in declared(plan.graph):
        if is_fixed_point(plan.graph[p]):
            plan = plan + Residualise(p)                                     # 2
    for block in Blocking.scc(plan.graph).blocks if fold else ():
        problems = [n for n in block if n in declared(plan.graph)]
        if len(problems) > 1:
            opt = next(n for n in problems if is_optimise(plan.graph[n]))
            problems.sort(key=lambda n: n != opt)                            # the optimiser leads the combined unknowns
            name = NodePath((GetAttrKey("sand_" + opt.leaf.name.removeprefix("Opt").lower()),))
            plan = plan + Combine(name, tuple(problems))                     # 3
    return plan

plan = build(SPLIT)
print("the recipe:")
print_recipe(plan)

blocking = Blocking.scc(plan.graph)
print("\nproblems, in run order:", [p.spelling for p in blocking.problems if p is not None])

the recipe:
   insert(.Constraint2, .Constraint16, .Constraint24, .Constraint8, .Constraint17, .Constraint18, .Constraint67,  ...
   residualise(^problem.physics.profiles.ion_vol_avg_temperature)
   residualise(^problem.power.delta_eta_step)
   residualise(^problem.physics.proton_rate_density.cycle)
   residualise(^problem.fwbs.f_ster_div_single)
   combine(^problem.sand_magnet <- .OptMagnet, ^problem.stellarator.coils.intersect)
   combine(^problem.sand_plasma <- .OptPlasma, ^problem.physics.profiles.ion_vol_avg_temperature, ^problem.physic ...



problems, in run order: ['^problem.sand_magnet', '^problem.sand_plasma', '^problem.power.delta_eta_step']


### Assign the drivers

VMCON for each stage and Newton for the root find outside them, attached in the graph.
The run order reads as two optimisations, magnet first and plasma second. The
power-conversion root find runs on its own after the plasma stage.

In [5]:
assigned = assign_drivers(plan.graph, default_drivers(plan.graph, bounds=design_bounds(ref.ixc)))
blocking = Blocking.scc(assigned)
schedule = Schedule(blocking)

for i, step in enumerate(schedule.steps):
    if isinstance(step, Drive):
        node = assigned[step.problem]
        print(f"step {i:3d}: Drive {step.problem.spelling:36s} {len(step.nodes):3d} nodes "
              f"{len(step.unknowns):2d} unknowns {len(step.conditions):2d} conditions  {type(node.driver).__name__}")
print(f"{sum(isinstance(s, Call) for s in schedule.steps)} Call steps, {len(schedule.steps)} steps in all")

step  25: Drive ^problem.sand_magnet                  25 nodes  5 unknowns  9 conditions  VmconDriver
step  47: Drive ^problem.sand_plasma                  85 nodes  8 unknowns 13 conditions  VmconDriver
step  48: Drive ^problem.power.delta_eta_step          3 nodes  1 unknowns  1 conditions  SeededNewtonDriver
60 Call steps, 63 steps in all


### What the graph refuses

The obvious alternative is machine `{2, 3, 4, 6, 10, 109}` against TF conductor
`{56, 59}`. It is not a legal split. COE depends on 56 and 59, and the conductor
constraints depend on 2 and 3, so both problems land in one loop. A loop with two
problems and no word on how they relate is refused before anything runs. The refusal
names the two ways out: merging, which gives one problem with two objectives that no
driver here answers, and nesting, which gives a bilevel problem.

In [6]:
SPLIT_B = {
    "OptMachine": ([2, 3, 4, 6, 10, 109], coe, [2, 16, 8, 17, 18, 24, 62, 67]),
    "OptTF":      ([56, 59], rmajor_obj, [32, 34, 35, 65, 82, 83]),
}
import re

try:
    Blocking.scc(build(SPLIT_B, fold=False).graph).problems
    print("scheduled -- unexpected")
except ValueError as refusal:                 # the node list dropped from the message
    print(re.sub(r"block \(.*?\) declares", "the block declares", str(refusal), flags=re.S))

the block declares several problems ((NodePath(^problem.stellarator.coils.intersect), NodePath(^problem.physics.profiles.ion_vol_avg_temperature), NodePath(^problem.physics.proton_rate_density.cycle), NodePath(^problem.fwbs.f_ster_div_single), NodePath(.OptMachine), NodePath(.OptTF))) with nothing saying which is outer -- one driver answers one problem, so `Combine` them into a single problem over every unknown, or `NestInside` one of them. Which is a modelling decision, not something the blocking can read off the graph


## 3. The process

The DSM in run order. The two optimiser boxes sit on the diagonal. Everything from the
magnet block to the plasma block lies below the diagonal, which is what "sequential"
looks like. The DSM is written as an interactive page next to this notebook. In the page, hover a
cell for the variables it carries and click a box to fold it.

In [7]:
from functional_process.cottax.render_xdsm import SPELLING
from functional_process.cottax.visualization.grouping import render_grouped_dsm_html, structure_order

dsm = render_grouped_dsm_html(
    blocking, order=structure_order(blocking),
    title="stellarator_helias -- two sequential Optimise (magnet, then plasma), run order",
    file_name="dsm_two_opt_driver", outdir=str(HERE), write=True, formatter=SPELLING,
)
print("written:", dsm.path)

Using adapted ragraph from debug branch


written: /home/wrutten/projects/functional_PROCESS/PROCESS/functional_process/architecture_examples/two_opt_driver/dsm_two_opt_driver.html


## 4. Run it

VMCON on each stage, recording each iterate, with each residual scaled by the size of
its quantity. Starting values follow `run_cold_matrix.solve_sand`: iteration variables
from the input file, every lifted unknown and every inner starting value from the
converged analysis. The schedule then runs step by step.

In [8]:
import dataclasses

from functional_process.cottax import mdf
from functional_process.cottax.core.solver.drivers import Status
from functional_process.cottax.mda import seed_starts
from functional_process.cottax.run_cold_matrix import _recorder, _trace_tail
from functional_process.cottax.run_sand_harness import _inputs_only, _seed
from functional_process.cottax.sand import residual_condition_scales
from functional_process.cottax.sand_harness import run_schedule

traces = {}
drivers = default_drivers(plan.graph, bounds=ref.bounds, max_iter=200)
for step in schedule.steps:
    if isinstance(step, Drive) and is_optimise(plan.graph[step.problem]):
        traces[step.problem] = []
        drivers[step.problem] = dataclasses.replace(
            drivers[step.problem],
            callback=_recorder(traces[step.problem]),
            condition_scale=residual_condition_scales(step, env),
        )
solve = Schedule(Blocking.scc(assign_drivers(plan.graph, drivers)))
drives = [s for s in solve.steps if isinstance(s, Drive) and s.problem in traces]

design_vars = {iteration_variable_path(i) for i in ref.ixc}
seeded = {}
for d in drives:
    e, _borrowed = _seed(solve, d, ref.cold, env, design=design_vars)
    seeded.update(e)
seeded.update(seed_starts(solve, env, exclude=design_vars))   # every start port from the MDA

began = time.perf_counter()
out = run_schedule(solve, _inputs_only(solve, seeded), whole=False)
print(f"solved in {time.perf_counter() - began:.1f} s (first call: includes compilation)\n")

iterations = {}
for d in drives:
    status = int(np.asarray(mdf.verdict(out, Status, d.problem)))
    n, objf, max_eq, min_ie = _trace_tail(traces[d.problem])
    iterations[d.problem.leaf.name] = n
    print(f"{d.problem.spelling:24s} status={status}  VMCON iterations={n:3d}  "
          f"max|eq|={max_eq:.1e}  min ineq={min_ie:+.1e}")
print("\ndesign:", {i: round(float(np.asarray(out[iteration_variable_path(i)])), 4) for i in ref.ixc})
print(f"b_t >= {BT_MIN}: condition value {float(np.asarray(out[cond('bt_min')])):+.2e} (active at 0)")
print(f"COE (plasma objective):    {float(np.asarray(out[coe])):.6f}")
print(f"rmajor (magnet objective): {float(np.asarray(out[rmajor_obj])):.4f}")

solved in 7.4 s (first call: includes compilation)

^problem.sand_magnet     status=0  VMCON iterations= 11  max|eq|=9.4e-08  min ineq=+5.1e-11
^problem.sand_plasma     status=0  VMCON iterations= 22  max|eq|=1.9e-11  min ineq=+4.1e-08

design: {2: 5.0, 3: 26.2914, 4: 5.7039, 6: 2.005171249824107e+20, 10: 1.0726, 56: 4.1496, 59: 0.3273, 109: 0.0805}
b_t >= 5.0: condition value -8.98e-10 (active at 0)
COE (plasma objective):    1.272169
rmajor (magnet objective): 5.2583


Both stages converge. For scale, the same file is solved as one joint SAND problem with
`session.open_session(...).sand()`. The sequential answer is feasible but not optimal,
because the magnet stage's conductor choice is paid for in the plasma stage.

In [9]:
from functional_process.cottax import session

live = session.open_session(INPUT)
sand_row = live.sand()
coe_two = float(np.asarray(out[coe]))
print(f"reference SAND: {sand_row['status']}, {sand_row['iterations']} iterations, COE {sand_row['objf']:.6f}")
print(f"two drivers:    {sum(iterations.values())} iterations in all, COE {coe_two:.6f} "
      f"({100 * (coe_two / sand_row['objf'] - 1):+.1f} % against the joint optimum)")

RESULT = {
    "iterations": iterations,
    "coe": coe_two,
    "design": {i: float(np.asarray(out[iteration_variable_path(i)])) for i in ref.ixc},
    "sand": {"status": sand_row["status"], "iterations": sand_row["iterations"], "coe": sand_row["objf"]},
}
RESULT

reference SAND: converged, 43 iterations, COE 1.218441
two drivers:    33 iterations in all, COE 1.272169 (+4.4 % against the joint optimum)


{'iterations': {'sand_magnet': 11, 'sand_plasma': 22},
 'coe': 1.2721691424383468,
 'design': {2: 5.000000004488335,
  3: 26.291430437147575,
  4: 5.703920572920458,
  6: 2.005171249824107e+20,
  10: 1.0726056888103634,
  56: 4.14962315602769,
  59: 0.32726504237970927,
  109: 0.08053231995089072},
 'sand': {'status': 'converged', 'iterations': 43, 'coe': 1.218441432897411}}

## What to take from it

- The split was **derived, not designed**. The dependency table says what a legal split
  can be, the run order comes out with the magnet stage first, and the illegal split is
  refused before anything runs.
- Legal says nothing about **feasible**. The split as designed leaves the plasma stage no
  feasible point. The `b_t` lower bound that recovers it is a hand-tuned interface with
  a price: COE a few percent above the joint optimum.
- Performance, the warm start of the full SAND from the two-stage answer, and the
  diagnosis of the infeasibility are in [`two_driver_report.md`](two_driver_report.md).